# Alice in Wonderland Next-Word Generator

A word-level Bidirectional LSTM trained on the first chapters of *Alice's Adventures in Wonderland*. The data is fed through a `tf.data` pipeline with a proper train/validation split, and the model then continues a sentence such as "Alice was".

**How to run:** Open in Google Colab, choose **Runtime > Change runtime type > GPU**, then **Runtime > Run all**.

_Rewritten from my original Colab notebook: `Alice was dataset model.ipynb`._

## 1. Imports

In [ ]:
import re
import urllib.request

import numpy as np
import tensorflow as tf
import keras
from keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)  # seeds Python, NumPy and TensorFlow
print("TensorFlow:", tf.__version__)

## 2. Download the book and split it into sentences

In [ ]:
def download_gutenberg_book(url):
    """Download a plain-text book from Project Gutenberg and remove the licence header/footer."""
    raw = urllib.request.urlopen(url).read().decode("utf-8-sig")
    start = raw.find("*** START OF")
    end = raw.find("*** END OF")
    if start != -1 and end != -1:
        raw = raw[raw.find("\n", start) + 1:end]
    return raw.replace("\r\n", "\n").strip()

In [ ]:
book = download_gutenberg_book("https://www.gutenberg.org/cache/epub/11/pg11.txt")
start = book.find("Alice was beginning")
text = book[start:start + 60000]  # roughly the first two chapters: enough to learn, quick to train
text = " ".join(text.split())  # join the wrapped lines

sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.split()) > 2]
print("Sentences:", len(sentences))
for s in sentences[:5]:
    print("-", s)

## 3. Vocabulary and n-gram samples
Long sentences are cut to the last 30 words to keep inputs small.

In [ ]:
def clean_words(line):
    """Lower-case a line and keep only words (letters and apostrophes)."""
    line = line.lower().replace("\u2019", "'")
    return re.findall(r"[a-z']+", line)


def build_vocab(lines):
    """Map every word to an integer id. Id 0 is reserved for padding."""
    words = sorted({w for line in lines for w in clean_words(line)})
    word_to_id = {w: i + 1 for i, w in enumerate(words)}
    id_to_word = {i: w for w, i in word_to_id.items()}
    return word_to_id, id_to_word


def encode(line, word_to_id):
    return [word_to_id[w] for w in clean_words(line) if w in word_to_id]

In [ ]:
def make_ngram_dataset(lines, word_to_id, max_len=None):
    """For each line 'a b c d' create the samples [a]->b, [a b]->c, [a b c]->d."""
    sequences = []
    for line in lines:
        ids = encode(line, word_to_id)
        for i in range(1, len(ids)):
            sequences.append(ids[: i + 1])
    if max_len is None:
        max_len = max(len(s) for s in sequences)
    padded = keras.utils.pad_sequences(sequences, maxlen=max_len, padding="pre", truncating="pre")
    X, y = padded[:, :-1], padded[:, -1]
    return X, y, max_len

In [ ]:
word_to_id, id_to_word = build_vocab(sentences)
vocab_size = len(word_to_id) + 1
X, y, max_len = make_ngram_dataset(sentences, word_to_id, max_len=30)
print("Samples:", len(X), "| vocabulary:", vocab_size)

## 4. Build a tf.data pipeline
We shuffle once (with a fixed seed) **before** splitting, so the train and validation sets never overlap.

In [ ]:
BATCH_SIZE = 64
indices = np.random.default_rng(SEED).permutation(len(X))
val_size = int(0.1 * len(X))
val_idx, train_idx = indices[:val_size], indices[val_size:]

train_ds = (tf.data.Dataset.from_tensor_slices((X[train_idx], y[train_idx]))
            .shuffle(len(train_idx), seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_ds = tf.data.Dataset.from_tensor_slices((X[val_idx], y[val_idx])).batch(BATCH_SIZE)

for xb, yb in train_ds.take(1):
    print("Batch shapes:", xb.shape, yb.shape)

## 5. Model

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(max_len - 1,)),
    layers.Embedding(vocab_size, 200),
    layers.Bidirectional(layers.LSTM(256, return_sequences=True)),
    layers.Dropout(0.3),
    layers.Bidirectional(layers.LSTM(256)),
    layers.Dropout(0.3),
    layers.Dense(vocab_size, activation="softmax"),
])
model.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 6. Train

In [ ]:
callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=5, min_lr=1e-5),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=100, callbacks=callbacks)

## 7. Evaluate

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="train loss")
if "val_loss" in history.history:
    plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("epoch")
plt.legend()
plt.show()

In [ ]:
loss, acc = model.evaluate(val_ds, verbose=0)
print(f"Validation loss: {loss:.3f} | accuracy: {acc:.3f}")

## 8. Predict

In [ ]:
def predict_next_word(model, seed_text, word_to_id, id_to_word, max_len):
    ids = encode(seed_text, word_to_id)
    padded = keras.utils.pad_sequences([ids], maxlen=max_len - 1, padding="pre", truncating="pre")
    probs = model.predict(padded, verbose=0)[0]
    probs[0] = 0  # never predict the padding id
    return id_to_word.get(int(np.argmax(probs)), "")


def generate_text(model, seed_text, n_words, word_to_id, id_to_word, max_len):
    text = seed_text
    for _ in range(n_words):
        next_word = predict_next_word(model, text, word_to_id, id_to_word, max_len)
        if not next_word:
            break
        text += " " + next_word
    return text

In [ ]:
for seed in ["Alice was", "the White Rabbit", "she was"]:
    print(generate_text(model, seed, 10, word_to_id, id_to_word, max_len))

In [ ]:
model.save("alice_text_generator.keras")